# Training data for the UW shallow cumulus kernel

`compute_uwshcu_inv` -- the CAM5 shallow convection scheme, the most expensive
kernel of the physics -- runs here as an ordinary Python function in a
standalone image linked from the pinned iCESM object code, so every number it
answers is the model's own arithmetic.

Two cells, as for `mmacro_pcond` (`generate_training_data.ipynb`).  The first
takes real atmospheric columns out of a capture of the running model; the
second draws samples around them and lets the Fortran answer each one.  Every
one of the kernel's 20 inputs is drawn for every sample -- the 57-constituent
tracer array included, its water isotopes kept on their water at the column's
own ratios -- and every one of its 30 outputs is written.

Every path is resolved from `site.env` at the repository root (see the
README's *Site configuration*); nothing below names a user.

## 1. Real atmospheric columns

A frame capture holds every argument of every call `compute_uwshcu_inv`
received in a run of the model: the month capture records one call in 25 on
each of 512 ranks, 61,440 calls, about 860,000 live columns of the model's
own state.  This draws `ANCHOR_COLUMNS` of them, all 20 arguments of a column
taken together so a sample stays one coherent atmospheric state.

An anchor file somebody published is found under `FREECAM_CAPTURE` in
`site.env` (`uwshcu_anchors_<N>.npz`); frame captures under your own scratch
are found too and the anchors drawn from them.  Making a new capture is a
512-rank run with the kernel's frames recorded:

    PYCAM_CAPTURE_KERNELS=compute_uwshcu_inv PYCAM_CAPTURE_EVERY=25 \
        validation/jobs/submit.sh validation/jobs/pi_cam_pausable_1month.pbs

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
import numpy as np

REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / 'pyproject.toml').is_file():
    REPO = REPO.parent
sys.path.insert(0, str(REPO / 'src'))
from freecam import site

SCRATCH = site.resolved(repo=REPO)['scratch']
WORK = SCRATCH / 'pyCAM' / 'uwshcu-training'; WORK.mkdir(parents=True, exist_ok=True)

FUNCTION, KERNEL = 'uwshcu', 'compute_uwshcu_inv'
ANCHOR_COLUMNS = int(os.environ.get('PYCAM_ANCHOR_COLUMNS', 200_000))

IMAGE = REPO / 'build/pi_cam_standalone' / FUNCTION / 'manifest.json'
assert IMAGE.is_file(), (
    f'no standalone image at {IMAGE}\n'
    f'  build one: tools/build_pi_cam_standalone_function.py --function {FUNCTION} --case <oracle case root>')

# A published anchor file first, the largest; then frame captures under your own scratch.
PUBLISHED = site.path('FREECAM_CAPTURE', repo=REPO)
published = sorted(PUBLISHED.glob(f'{FUNCTION}_anchors_*.npz'), key=lambda p: p.stat().st_size) if PUBLISHED else []
captures = sorted(SCRATCH.glob(f'pyCAM/PI-cam/*{KERNEL.split("_")[1]}*capture*/frame-capture'))

ANCHORS = WORK / f'anchors_{ANCHOR_COLUMNS}.npz'
if not ANCHORS.is_file():
    if published:
        source = published[-1]
        held = int(json.loads(str(np.load(source, allow_pickle=True)['provenance']))['columns'])
        if held >= ANCHOR_COLUMNS:
            ANCHORS = source                    # use it as it is
        else:
            raise SystemExit(f'{source} holds {held:,} anchors; asked for {ANCHOR_COLUMNS:,}')
    elif captures:
        print(f'extracting {ANCHOR_COLUMNS:,} anchors from {captures[-1]}', flush=True)
        subprocess.run([sys.executable, str(REPO / 'tools/extract_pi_cam_anchor_columns.py'),
                        '--function', FUNCTION, '--kernel', KERNEL, '--frame-capture', str(captures[-1]),
                        '--columns', str(ANCHOR_COLUMNS), '--output', str(ANCHORS)], check=True, stdout=subprocess.DEVNULL)
    else:
        raise SystemExit('no anchors: point FREECAM_CAPTURE in site.env at a published uwshcu_anchors_<N>.npz, '
                         'or make a frame capture (see above)')

anchors = np.load(ANCHORS, allow_pickle=True)
where = json.loads(str(anchors['provenance']))
print(f"{where['columns']:,} anchors from {where['live_columns']:,} captured columns")
print(f"  capture: {where.get('frame_capture', where.get('bundle'))}")
print(f"  {len([n for n in anchors.files if not n.startswith('meta_') and n != 'provenance'])} arguments each, "
      f"{np.asarray(anchors['t0_inv']).shape[1]} levels, {np.asarray(anchors['tr0_inv']).shape[2]} constituents")
print(f'  {ANCHORS}  ({ANCHORS.stat().st_size / 1e9:.2f} GB)')

## 2. Samples

Each sample draws one anchor column, perturbs it at a stated budget, rebuilds
what must stay consistent (the static energy with the temperature, the tracer
array with the water, the convective cloud fraction within the cloud fraction,
the cumulus scale height where there is no cumulus), draws the timestep and the
penetrative entrainment efficiency `uwshcu_rpen`, and calls the Fortran for the
answer.  The rules are written out in the script's `SAMPLING_NOTES` and land in
the file's attributes.

The work is split across processes; each holds only its share of the anchors.

In [ ]:
SAMPLES = int(os.environ.get('PYCAM_SAMPLES', 200_000))
CHUNKS = int(os.environ.get('PYCAM_CHUNKS', 10))
per_chunk = SAMPLES // CHUNKS
print(f'{SAMPLES:,} samples in {CHUNKS} processes of {per_chunk:,}', flush=True)

started = time.monotonic()
running, files = [], []
for index in range(CHUNKS):
    out = WORK / f'chunk_{index}.nc'
    files.append(out)
    if out.is_file():
        continue
    running.append(subprocess.Popen(
        [sys.executable, str(REPO / 'examples/generate_compute_uwshcu_inv_dataset.py'),
         '--samples', str(per_chunk), '--seed', str(2026 + index),
         '--anchor-bundle', str(ANCHORS), '--output', str(out),
         '--anchor-part', str(index), '--anchor-parts', str(CHUNKS)],
        stdout=subprocess.DEVNULL, stderr=subprocess.PIPE))
for process in running:
    if process.wait() != 0:
        raise SystemExit(process.stderr.read().decode()[-2000:])
print(f'{len(files)} files in {time.monotonic() - started:.0f} s:')
for out in files:
    print(f'  {out}  ({out.stat().st_size / 1e6:.0f} MB)')

from freecam.physics import open_dataset
first = open_dataset(files[0])
print(first)
print('inputs :', ', '.join(first.inputs))
print('outputs:', ', '.join(first.outputs), '| updated:', ', '.join(first.updated))
print('status :', dict(zip(*np.unique(first.status, return_counts=True))))

## What to do with it

The chunks are ordinary NetCDF files: `input__<name>`, `output__<name>` and
`updated__<name>` over `sample` and the contract's axes (`lev`, `ilev`,
`dim_57` for the constituents), `parameter__uwshcu_rpen`, `status`,
`sample_id`, and the sampling notes in the attributes.  `open_dataset` reads
one; `verify_sample(scheme)` re-runs a stored sample through the Fortran and
checks it comes back identical.

A model trained on them goes into the running model at the kernel's slot as a
TorchScript file: `driver.processes['shallow_convection'].kernels['compute_uwshcu_inv'] = fc.NativeModel('model.pt')`
(see `docs/usage.md`, *Replacing one kernel inside a process*, and
`docs/contracts.md` for the packed block the model returns).  Split training
and held-out rows by anchor, not at random: samples reuse anchors, so a random
split puts variants of one atmospheric state on both sides.